In [3]:
pip install sdv sdmetrics

Note: you may need to restart the kernel to use updated packages.


*Libreria SDV*
*Autor Brenda Alexandra Martinez Rivera*
*Fecha 04 de Junio*

In [4]:
import pandas as pd
import numpy as np
import random

In [6]:
import sdv
import sdmetrics
from sdv.metadata import SingleTableMetadata
from sdv.single_table import GaussianCopulaSynthesizer

In [8]:
from sdmetrics.reports.single_table import QualityReport

In [10]:
print ("SDV: ", sdv.__version__)

SDV:  1.37.0


In [12]:
dfClientes = pd.DataFrame(
    {
        "cliente_id":[1,2,3,4,5,6,7,8,9,10],
        "edad": [23,33,43,28,53,56,43,56,65,40],
        "ingreso_mensual": [25000,15000,20000,10000.5000,17000,30000,12000,35000,7500,18000],
        "ciudad": ["Veracruz","Cordoba","Paso del macho","Amatlan","Fortin","Cuitlahuac","Cordoba","Yanga","Orizaba","Cuitlahuac"],
        
    }
)

In [14]:
dfClientes.head()

,cliente_id,edad,ingreso_mensual,ciudad
0,1,23,25000.0,Veracruz
1,2,33,15000.0,Cordoba
2,3,43,20000.0,Paso del macho
3,4,28,10000.5,Amatlan
4,5,53,17000.0,Fortin


In [15]:
metadata = SingleTableMetadata()

In [16]:
metadata.detect_from_dataframe(
    data = dfClientes
)

In [ ]:
#Instalar graphviz (URL winget)
metadata.visualize()

In [19]:
metadata.to_dict()

{'METADATA_SPEC_VERSION': 'SINGLE_TABLE_V1',
 'primary_key': 'cliente_id',
 'columns': {'cliente_id': {'sdtype': 'id'},
  'edad': {'sdtype': 'numerical'},
  'ingreso_mensual': {'sdtype': 'numerical'},
  'ciudad': {'sdtype': 'categorical'}}}

In [20]:
#Gaurdamos el metadata en un archivo texto
metadata.save_to_json(
    "dfClientes_metadata.json"
)

In [21]:
#Entrar el módelo para generar los datos sintéticos
synthesizer = GaussianCopulaSynthesizer (
metadata
)

C:\Users\brenr\anaconda3\Lib\site-packages\sdv\single_table\base.py:182: FutureWarning:

The 'SingleTableMetadata' is deprecated. Please use the new 'Metadata' class for synthesizers.



In [22]:
#Entrenamiento
synthesizer.fit(
    dfClientes
)

In [23]:
#Generamos los datos sinteticos
clientes_sinteticos = synthesizer.sample(
    num_rows = 100
)

In [24]:
clientes_sinteticos.head()

,cliente_id,edad,ingreso_mensual,ciudad
0,16169768,51,11513.5,Yanga
1,4918803,47,27803.9,Fortin
2,1900081,36,16819.6,Fortin
3,531516,34,25358.0,Fortin
4,10211768,54,18899.0,Yanga


In [25]:
clientes_sinteticos.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   cliente_id       100 non-null    int64  
 1   edad             100 non-null    int64  
 2   ingreso_mensual  100 non-null    float64
 3   ciudad           100 non-null    object 
dtypes: float64(1), int64(2), object(1)
memory usage: 3.3+ KB


In [26]:
clientes_sinteticos.describe(include = "all")

,cliente_id,edad,ingreso_mensual,ciudad
count,1.000000e+02,100.000000,100.000000,100
unique,NaN,NaN,NaN,8
top,NaN,NaN,NaN,Cordoba
freq,NaN,NaN,NaN,24
mean,8.532170e+06,42.770000,20006.780000,NaN
std,4.687773e+06,11.542504,7499.675348,NaN
min,4.660500e+04,28.000000,10185.500000,NaN
25%,5.008268e+06,32.000000,13332.450000,NaN
50%,8.656048e+06,41.000000,18997.500000,NaN
75%,1.256688e+07,52.250000,26242.275000,NaN


In [27]:
#DataFrame contaminado
dfClientesGIGO = clientes_sinteticos.copy()

In [28]:
#Colocamos edades imposibles
indices = random.sample(list(dfClientesGIGO.index),5)

In [29]:
dfClientesGIGO.loc[indices, "edad"] = -5 

In [34]:
#Introducir valores nulos
duplicados = dfClientesGIGO.sample(10, random_state=42)

In [35]:
dfClientesGIGO = pd.concat([dfClientesGIGO, duplicados], ignore_index=True)

In [36]:
dfClientesGIGO.duplicated().sum()

np.int64(10)

In [37]:
dfClientesGIGO.describe(include = "all")

,cliente_id,edad,ingreso_mensual,ciudad
count,1.100000e+02,110.000000,110.000000,110
unique,NaN,NaN,NaN,8
top,NaN,NaN,NaN,Cordoba
freq,NaN,NaN,NaN,28
mean,8.614045e+06,39.754545,20094.012727,NaN
std,4.775996e+06,14.940976,7412.335624,NaN
min,4.660500e+04,-5.000000,10185.500000,NaN
25%,4.948624e+06,30.000000,13401.275000,NaN
50%,8.956428e+06,38.000000,19096.000000,NaN
75%,1.274153e+07,51.750000,26002.125000,NaN


In [38]:
#Gnerar el reporte
report = QualityReport()

In [40]:
report.generate(
    real_data = dfClientes,
    synthetic_data = clientes_sinteticos,
    metadata = metadata.to_dict()
)

Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 4/4 [00:00<00:00, 114.62it/s]|
Column Shapes Score: 84.67%

(2/2) Evaluating Column Pair Trends: |██████████| 6/6 [00:00<00:00, 19.50it/s]|
Column Pair Trends Score: 18.5%

Overall Score (Average): 51.58%



In [41]:
report.generate(
    real_data = dfClientes,
    synthetic_data = dfClientesGIGO,
    metadata = metadata.to_dict()
)

Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 4/4 [00:00<00:00, 26.29it/s]|
Column Shapes Score: 82.42%

(2/2) Evaluating Column Pair Trends: |██████████| 6/6 [00:00<00:00, 16.78it/s]|
Column Pair Trends Score: 13.64%

Overall Score (Average): 48.03%

